In [ ]:
import sys
from pathlib import Path
ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

import pandas as pd
from src.config import *
from src.preprocessing import (
    preprocess_daily,
    preprocess_monthly_to_daily,
    merge_datasets
)

ihsg = pd.read_csv(IHSG_FILE)
kurs = pd.read_csv(KURS_FILE)
bi = pd.read_csv(BI_RATE_FILE)
inflasi = pd.read_csv(INFLASI_FILE)
pdb = pd.read_csv(PDB_FILE)

In [12]:
print(ihsg.head())

         date     ihsg
0  30-12-2024  7079.90
1  27-12-2024  7036.57
2  24-12-2024  7065.75
3  23-12-2024  7096.44
4  20-12-2024  6983.87


In [13]:
# Daily data
ihsg_clean = preprocess_daily(ihsg, "date", ["ihsg"])
kurs_clean = preprocess_daily(kurs, "date", ["nilai_tukar"])

D:\project_skripsi\src\preprocessing.py:15: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[date_col] = pd.to_datetime(df[date_col])
D:\project_skripsi\src\preprocessing.py:15: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[date_col] = pd.to_datetime(df[date_col])


In [ ]:
# Monthly -> daily
bi_daily = preprocess_monthly_to_daily(bi, "date", "bi_rate")
inflasi_daily = preprocess_monthly_to_daily(inflasi, "date", "inflasi")
pdb_daily = preprocess_monthly_to_daily(pdb, "date", "pdb")

In [16]:
bi_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3623 entries, 0 to 3622
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     3623 non-null   datetime64[ns]
 1   bi_rate  3623 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 56.7 KB


In [17]:
inflasi_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3623 entries, 0 to 3622
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     3623 non-null   datetime64[ns]
 1   inflasi  3623 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 56.7 KB


In [19]:
pdb_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3623 entries, 0 to 3622
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    3623 non-null   datetime64[ns]
 1   PDB     3623 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 56.7 KB


In [27]:
def save_dataframe(df, path, filename):
    """
    Simpan dataframe ke CSV
    """
    full_path = path / filename
    df.to_csv(full_path, index=False)
    print(f"Saved: {full_path}")

save_dataframe(bi_daily, DATA_RAW, "bi_daily.csv")
save_dataframe(inflasi_daily, DATA_RAW, "inflasi_daily.csv")
save_dataframe(pdb_daily, DATA_RAW, "pdb_daily.csv")

Saved: D:\project_skripsi\data\raw\bi_daily.csv
Saved: D:\project_skripsi\data\raw\inflasi_daily.csv
Saved: D:\project_skripsi\data\raw\pdb_daily.csv


In [28]:
base = ihsg_clean.merge(kurs_clean, on="date", how="left")

final_df = merge_datasets(
    base,
    [bi_daily, inflasi_daily, pdb_daily]
)

In [29]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2426 entries, 0 to 2425
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         2426 non-null   datetime64[ns]
 1   ihsg         2426 non-null   float64       
 2   nilai_tukar  2426 non-null   float64       
 3   bi_rate      2407 non-null   float64       
 4   inflasi      2407 non-null   float64       
 5   PDB          2407 non-null   float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 113.8 KB


In [31]:
final_df.tail()

,date,ihsg,nilai_tukar,bi_rate,inflasi,PDB
2421,2024-12-20,6983.87,16190.0,NaN,NaN,NaN
2422,2024-12-23,7096.44,16170.0,NaN,NaN,NaN
2423,2024-12-24,7065.75,16185.0,NaN,NaN,NaN
2424,2024-12-27,7036.57,16230.0,NaN,NaN,NaN
2425,2024-12-30,7079.90,16130.0,NaN,NaN,NaN


In [33]:
master_dates = ihsg_clean[["date"]].copy()

In [34]:
kurs_aligned = (
    master_dates
    .merge(kurs_clean, on="date", how="left")
    .ffill()
)

In [ ]:
bi_aligned = (
    master_dates
    .merge(bi_daily, on="date", how="left")
    .ffill()
)

inflasi_aligned = (
    master_dates
    .merge(inflasi_daily, on="date", how="left")
    .ffill()
)

pdb_aligned = (
    master_dates
    .merge(pdb_daily, on="date", how="left")
    .ffill()
)

In [36]:
final_df = master_dates.copy()

final_df = final_df.merge(ihsg_clean, on="date", how="left")
final_df = final_df.merge(kurs_aligned[["date", "nilai_tukar"]], on="date", how="left")
final_df = final_df.merge(bi_aligned[["date", "bi_rate"]], on="date", how="left")
final_df = final_df.merge(inflasi_aligned[["date", "inflasi"]], on="date", how="left")
final_df = final_df.merge(pdb_aligned[["date", "PDB"]], on="date", how="left")

In [37]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2426 entries, 0 to 2425
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         2426 non-null   datetime64[ns]
 1   ihsg         2426 non-null   float64       
 2   nilai_tukar  2426 non-null   float64       
 3   bi_rate      2426 non-null   float64       
 4   inflasi      2426 non-null   float64       
 5   PDB          2426 non-null   float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 113.8 KB


In [39]:
final_df.tail()

,date,ihsg,nilai_tukar,bi_rate,inflasi,PDB
2421,2024-12-20,6983.87,16190.0,6.0,0.3,3296741.7
2422,2024-12-23,7096.44,16170.0,6.0,0.3,3296741.7
2423,2024-12-24,7065.75,16185.0,6.0,0.3,3296741.7
2424,2024-12-27,7036.57,16230.0,6.0,0.3,3296741.7
2425,2024-12-30,7079.90,16130.0,6.0,0.3,3296741.7


In [43]:
final_df["pdb"] = final_df["PDB"] / 1_000_000
final_df = final_df.drop(columns=["PDB"])

In [44]:
final_df.tail()

,date,ihsg,nilai_tukar,bi_rate,inflasi,pdb
2421,2024-12-20,6983.87,16190.0,6.0,0.3,3.296742
2422,2024-12-23,7096.44,16170.0,6.0,0.3,3.296742
2423,2024-12-24,7065.75,16185.0,6.0,0.3,3.296742
2424,2024-12-27,7036.57,16230.0,6.0,0.3,3.296742
2425,2024-12-30,7079.90,16130.0,6.0,0.3,3.296742


In [45]:
final_df.to_csv(DATA_PROCESSED / "dataset_final.csv", index=False)

In [1]:
import pandas as pd

final_df = pd.read_csv("../data/processed/dataset_final.csv", parse_dates=["date"])

In [3]:
final_df.head()

,date,ihsg,nilai_tukar,bi_rate,inflasi,pdb
0,2015-01-02,5242.77,12542.5,7.75,-0.24,2.15804
1,2015-01-05,5219.99,12627.5,7.75,-0.24,2.15804
2,2015-01-06,5169.06,12657.5,7.75,-0.24,2.15804
3,2015-01-07,5207.12,12738.5,7.75,-0.24,2.15804
4,2015-01-08,5211.83,12680.0,7.75,-0.24,2.15804


In [7]:
final_df = final_df.rename(columns={
    "bi_rate": "suku_bunga" , "kurs_rupiah" : "nilai_tukar"
})

final_df.head()

,date,ihsg,nilai_tukar,suku_bunga,inflasi,pdb
0,2015-01-02,5242.77,12542.5,7.75,-0.24,2.15804
1,2015-01-05,5219.99,12627.5,7.75,-0.24,2.15804
2,2015-01-06,5169.06,12657.5,7.75,-0.24,2.15804
3,2015-01-07,5207.12,12738.5,7.75,-0.24,2.15804
4,2015-01-08,5211.83,12680.0,7.75,-0.24,2.15804


In [9]:
final_df.to_csv("../data/processed/dataset_final.csv", index=False)